# OpenRAN POWDER Testbed — UE Data Analysis
**Setup**: 1 Open5GS core (pc811) · 1 gNB (pc818, 50 srsenb, ZMQ) · 49 UEs (pc808)  
**Radio**: srsRAN 4G · ZMQ · 10 MHz · 50 PRB · FDD  

### Metrics collected
- UDP throughput ramp: 1, 10, 20, 50, 100, 200, 300, 400, 500 Mbps per UE
- ICMP latency (idle + under 100 Mbps load)
- RAN: PUCCH SNR, CQI, PUSCH SNR, MCS UL/DL, PRB utilization, Timing Advance
- System: CPU%, RAM, temperatures, softirq rates, context switches

### Effective Throughput Definition
```
effective_tput_mbps = reported_tput_mbps × (1 − packet_loss / 100)
```
Note: ZMQ is a perfect software channel — loss is from radio scheduler PRB limits, not RF noise.

In [ ]:
# ── 1. Install & imports ──────────────────────────────────────────────────────
!pip install matplotlib numpy pandas seaborn -q

import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from collections import defaultdict

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f9fafb',
    'axes.grid': True, 'grid.color': '#e5e7eb', 'grid.linewidth': 0.6,
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'xtick.labelsize': 8, 'ytick.labelsize': 9, 'legend.fontsize': 9,
})
BLUE='#3b82d4'; GREEN='#22c55e'; ORANGE='#f97316'; RED='#ef4444'; PURPLE='#7c3aed'
print('✓ imports done')

In [ ]:
# ── 2. Load embedded data ─────────────────────────────────────────────────────
RAW = __DATA_JSON__

# ── UDP measurements ──────────────────────────────────────────────────────────
# Columns: ue_id, rate_target_mbps, throughput_mbps, loss_pct, jitter_ms
udp_df = pd.DataFrame(RAW['udp'],
    columns=['ue_id','rate_target','tput_mbps','loss_pct','jitter_ms'])
udp_df['effective_mbps'] = udp_df['tput_mbps'] * (1 - udp_df['loss_pct']/100)

# ── ICMP latency ──────────────────────────────────────────────────────────────
# Columns: ue_id, min_ms, avg_ms, max_ms, mdev_ms
ping_df = pd.DataFrame(RAW['ping'],
    columns=['ue_id','ping_min_ms','ping_avg_ms','ping_max_ms','ping_mdev_ms'])

# ── RAN metrics ───────────────────────────────────────────────────────────────
# Columns: ue_id, pucch_snr, cqi, ta_us, pusch_snr, pusch_mcs, pusch_tbs,
#          pdsch_prb, prb_util_pct, pdsch_mcs, rss_kB, cpu_max, cpu_mean,
#          dl_brate, ul_brate
ran_df = pd.DataFrame(RAW['ran'],
    columns=['ue_id','pucch_snr','cqi','ta_us','pusch_snr','pusch_mcs','pusch_tbs',
             'pdsch_prb','prb_util_pct','pdsch_mcs','rss_kB','cpu_max','cpu_mean',
             'dl_brate','ul_brate'])

RATES=[1,10,20,50,100,200,300,400,500]
print(f'UDP rows: {len(udp_df)}  |  UEs: {udp_df.ue_id.nunique()}')
print(f'Ping rows: {len(ping_df)}  |  UEs: {ping_df.ue_id.nunique()}')
print(f'RAN rows: {len(ran_df)}')
print()
print('UDP sample:')
display(udp_df.head(3))
print('RAN sample:')
display(ran_df.head(3))

In [ ]:
# ── 3. Effective Throughput = Reported × (1 − Loss%) ─────────────────────────
# Plot 1: Sent vs Reported vs Effective — grouped bars per rate

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(RATES)); w = 0.28

sent  = RATES
recv  = [udp_df[udp_df.rate_target==r]['tput_mbps'].mean() for r in RATES]
eff   = [udp_df[udp_df.rate_target==r]['effective_mbps'].mean() for r in RATES]

ax.bar(x-w,   sent, width=w, color='#94a3b8', edgecolor='white', label='Target (sent) Mbps')
ax.bar(x,     recv, width=w, color=BLUE,       edgecolor='white', label='Reported throughput')
ax.bar(x+w,   eff,  width=w, color=GREEN,      edgecolor='white', label='Effective (after loss)')

for xi, ev in enumerate(eff):
    ax.text(xi+w, ev+1, f'{ev:.1f}', ha='center', fontsize=8, color='#166534', fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels([f'{r}M' for r in RATES])
ax.set_xlabel('Target UDP Rate'); ax.set_ylabel('Throughput (Mbps)')
ax.set_title('Sent vs Reported vs Effective Throughput per Rate\n'
             'Effective = Reported × (1 − Loss%)')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── 4. Loss vs Throughput Trade-off Curve ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: mean loss% vs target rate
ax = axes[0]
mean_loss = [udp_df[udp_df.rate_target==r]['loss_pct'].mean() for r in RATES]
ax.plot(RATES, mean_loss, color=RED, marker='o', ms=8, lw=2)
ax.fill_between(RATES, mean_loss, alpha=0.15, color=RED)
for r, l in zip(RATES, mean_loss):
    ax.annotate(f'{l:.0f}%', (r, l), textcoords='offset points', xytext=(0, 7),
                ha='center', fontsize=8.5, color='#b91c1c')
ax.set_xscale('log'); ax.set_xticks(RATES)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel('Target Rate (Mbps)'); ax.set_ylabel('Mean Packet Loss %')
ax.set_title('Packet Loss vs Target Rate')
ax.set_ylim(0, 105)

# Right: reported vs effective tput
ax = axes[1]
mean_recv = [udp_df[udp_df.rate_target==r]['tput_mbps'].mean()    for r in RATES]
mean_eff  = [udp_df[udp_df.rate_target==r]['effective_mbps'].mean() for r in RATES]
ax.plot(RATES, mean_recv, color=BLUE,  marker='o', ms=8, lw=2, label='Reported tput')
ax.plot(RATES, mean_eff,  color=GREEN, marker='s', ms=8, lw=2, label='Effective tput')
ax.fill_between(RATES, mean_recv, mean_eff, alpha=0.12, color=ORANGE,
                label='Loss gap')
ax.set_xscale('log'); ax.set_xticks(RATES)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel('Target Rate (Mbps)'); ax.set_ylabel('Throughput (Mbps)')
ax.set_title('Reported vs Effective Throughput')
ax.legend()

plt.suptitle('Loss vs Throughput Analysis — All UEs', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 5. Per-UE Throughput at 100 Mbps — bar chart ─────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(18, 9))

sub = udp_df[udp_df.rate_target == 100].sort_values('ue_id')
ue_ids = sub['ue_id'].tolist()
x = np.arange(len(ue_ids))

# Top: Reported throughput
ax = axes[0]
ax.bar(x, sub['tput_mbps'], color=BLUE, edgecolor='white', width=0.75)
ax.axhline(100, color=ORANGE, ls='--', lw=1.3, label='Target 100 Mbps')
ax.set_xticks(x); ax.set_xticklabels([f'UE{u}' for u in ue_ids], rotation=45, ha='right', fontsize=7.5)
ax.set_ylabel('Reported Throughput (Mbps)')
ax.set_title('Reported UDP Throughput @ 100 Mbps — All UEs')
ax.legend()

# Bottom: Effective throughput
ax = axes[1]
colors = [GREEN if e > 0.5 else RED for e in sub['effective_mbps']]
ax.bar(x, sub['effective_mbps'], color=colors, edgecolor='white', width=0.75)
ax.set_xticks(x); ax.set_xticklabels([f'UE{u}' for u in ue_ids], rotation=45, ha='right', fontsize=7.5)
ax.set_ylabel('Effective Throughput (Mbps)')
ax.set_title('Effective UDP Throughput @ 100 Mbps — Reported × (1 − Loss%)')

plt.tight_layout(); plt.show()

In [ ]:
# ── 6. Box plots — distribution at each rate ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribution Across All UEs at Each Rate', fontsize=13, fontweight='bold')

metrics = [
    ('tput_mbps',     'Reported Throughput (Mbps)', BLUE),
    ('effective_mbps','Effective Throughput (Mbps)', GREEN),
    ('loss_pct',      'Packet Loss %', RED),
]
for (col, ylabel, c), ax in zip(metrics, axes):
    data = [udp_df[udp_df.rate_target==r][col].dropna().tolist() for r in RATES]
    bp = ax.boxplot(data, patch_artist=True,
                    tick_labels=[f'{r}M' for r in RATES], widths=0.5)
    for patch in bp['boxes']:   patch.set_facecolor(c); patch.set_alpha(0.65)
    for m in bp['medians']:     m.set_color('black'); m.set_linewidth(2.5)
    for f in bp['fliers']:      f.set(marker='o', markersize=4, alpha=0.5)
    ax.set_xlabel('Target Rate'); ax.set_ylabel(ylabel)
    ax.set_title(ylabel)

plt.tight_layout(); plt.show()

In [ ]:
# ── 7. Scatter — Loss% vs Effective Throughput, coloured by rate ──────────────
fig, ax = plt.subplots(figsize=(11, 6))
cmap = plt.cm.viridis
norm = mcolors.LogNorm(vmin=1, vmax=500)

for rate in RATES:
    sub = udp_df[udp_df.rate_target == rate]
    c = cmap(norm(rate))
    ax.scatter(sub['effective_mbps'], sub['loss_pct'],
               color=c, alpha=0.65, s=40, edgecolors='none', label=f'{rate}M')

ax.set_xlabel('Effective Throughput (Mbps)')
ax.set_ylabel('Packet Loss %')
ax.set_title('Effective Throughput vs Packet Loss\n'
             'Each dot = one UE measurement (colour = target rate)')
ax.legend(title='Target Rate', ncol=3, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── 8. ICMP Latency — bar + error per UE ──────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(18, 9))

ping = ping_df.sort_values('ue_id')
x = np.arange(len(ping))

# Top: avg RTT with error bars (mdev)
ax = axes[0]
ax.bar(x, ping['ping_avg_ms'], color=BLUE, edgecolor='white', width=0.7)
ax.errorbar(x, ping['ping_avg_ms'], yerr=ping['ping_mdev_ms'],
            fmt='none', ecolor=ORANGE, capsize=3, lw=1.5, label='±jitter (mdev)')
ax.set_xticks(x); ax.set_xticklabels([f'UE{u}' for u in ping['ue_id']],
                                       rotation=45, ha='right', fontsize=7.5)
ax.set_ylabel('Avg RTT (ms)')
ax.set_title('ICMP Latency per UE — Idle Baseline (bars=avg, error=jitter)')
ax.legend()

# Bottom: min/avg/max
ax = axes[1]
w = 0.28
ax.bar(x-w,   ping['ping_min_ms'], width=w, color=GREEN,  edgecolor='white', label='Min RTT')
ax.bar(x,     ping['ping_avg_ms'], width=w, color=BLUE,   edgecolor='white', label='Avg RTT')
ax.bar(x+w,   ping['ping_max_ms'], width=w, color=RED,    edgecolor='white', label='Max RTT')
ax.set_xticks(x); ax.set_xticklabels([f'UE{u}' for u in ping['ue_id']],
                                       rotation=45, ha='right', fontsize=7.5)
ax.set_ylabel('RTT (ms)')
ax.set_title('Min / Avg / Max ICMP RTT per UE')
ax.legend()

plt.tight_layout(); plt.show()

In [ ]:
# ── 9. RAN Radio Metrics — one bar per metric ─────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('RAN Radio Metrics — 50 UE Slots (gnb1)', fontsize=13, fontweight='bold')

ran = ran_df.sort_values('ue_id').reset_index(drop=True)
x = np.arange(len(ran))
ue_labels = [f'UE{u}' for u in ran['ue_id']]

plots = [
    ('pucch_snr', 'PUCCH SNR (dB)',        BLUE,   (0, 130)),
    ('cqi',       'CQI (0–15)',             GREEN,  (0, 17)),
    ('prb_util_pct','PRB Utilization %',   PURPLE, (0, 100)),
    ('pusch_snr', 'PUSCH SNR (dB)',         ORANGE, (0, 130)),
    ('pusch_mcs', 'UL MCS',                 RED,    (0, 30)),
    ('pdsch_mcs', 'DL MCS',                 '#06b6d4', (0, 30)),
]
for (col, title, c, ylim), ax in zip(plots, axes.flat):
    vals = ran[col].fillna(0)
    ax.bar(x, vals, color=c, edgecolor='white', width=0.75)
    ax.set_xticks(x[::5]); ax.set_xticklabels(ue_labels[::5], rotation=45, ha='right', fontsize=7)
    ax.set_ylim(*ylim); ax.set_title(title); ax.set_ylabel(title.split('(')[0].strip())

plt.tight_layout(); plt.show()

In [ ]:
# ── 10. Effective Throughput Heatmap (UE × Rate) ─────────────────────────────
pivot = udp_df.pivot_table(index='ue_id', columns='rate_target',
                            values='effective_mbps', aggfunc='mean')
pivot = pivot.reindex(columns=RATES)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlGn', vmin=0, vmax=25)
ax.set_xticks(range(len(RATES))); ax.set_xticklabels([f'{r}M' for r in RATES])
ax.set_yticks(range(len(pivot))); ax.set_yticklabels([f'UE{u}' for u in pivot.index], fontsize=7)
ax.set_xlabel('Target UDP Rate'); ax.set_ylabel('UE')
ax.set_title('Effective Throughput Heatmap — All UEs × All Rates\n'
             'Effective = Reported × (1−Loss%), scale capped at 25 Mbps')
cb = plt.colorbar(im, ax=ax, shrink=0.6); cb.set_label('Effective Tput (Mbps)')
plt.tight_layout(); plt.show()

print('\nMean effective throughput per rate:')
print(udp_df.groupby('rate_target')['effective_mbps'].mean().round(3))

In [ ]:
# ── 11. CPU & RAM per UE Slot (gnb1 srsenb) ───────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(18, 9))

ran_valid = ran_df[ran_df['cpu_max'].notna()].sort_values('ue_id')
x = np.arange(len(ran_valid))

ax = axes[0]
w = 0.38
ax.bar(x-w/2, ran_valid['cpu_max'],  width=w, color=BLUE,  edgecolor='white', label='CPU Max %')
ax.bar(x+w/2, ran_valid['cpu_mean'], width=w, color=GREEN, edgecolor='white', label='CPU Mean %')
ax.set_xticks(x); ax.set_xticklabels([f'UE{u}' for u in ran_valid['ue_id']],
                                       rotation=45, ha='right', fontsize=7.5)
ax.set_ylabel('CPU %'); ax.set_title('srsenb CPU Usage per UE Slot (gnb1)')
ax.legend()

ax = axes[1]
rss_mb = ran_valid['rss_kB'] / 1024
ax.bar(x, rss_mb, color=PURPLE, edgecolor='white', width=0.75)
ax.axhline(rss_mb.mean(), color=ORANGE, ls='--', lw=1.5,
           label=f'Mean RSS = {rss_mb.mean():.0f} MB')
ax.set_xticks(x); ax.set_xticklabels([f'UE{u}' for u in ran_valid['ue_id']],
                                       rotation=45, ha='right', fontsize=7.5)
ax.set_ylabel('Process RSS (MB)'); ax.set_title('srsenb Process RAM (RSS) per UE Slot')
ax.legend()

plt.tight_layout(); plt.show()

In [ ]:
# ── 12. Summary statistics table ──────────────────────────────────────────────
print('=' * 60)
print('  SUMMARY STATISTICS')
print('=' * 60)

print('\n── UDP Throughput ──')
display(udp_df.groupby('rate_target')[['tput_mbps','effective_mbps','loss_pct','jitter_ms']]
         .agg(['mean','min','max']).round(2))

print('\n── ICMP Latency ──')
display(ping_df[['ping_min_ms','ping_avg_ms','ping_max_ms','ping_mdev_ms']]
         .describe().round(1))

print('\n── RAN Metrics ──')
display(ran_df[['pucch_snr','cqi','prb_util_pct','pusch_mcs','pdsch_mcs']]
         .describe().round(2))